# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
contract_summary = {
    "unit_grain": "One row represents one pseudonymized content item (content_id) for a specific client during a specific time period.",
    "tables_used": "dim_content joined with fact_content_daily_performance, filtered to month=2026-03.",
    "time_window": "Mid-panel month 2026-03 (March 2026) for training and feature generation; June 2026 is excluded as the sealed test month.",
    "target_proxy": "Binary proxy is_declining_label: 1 if the 30-day impression trend is down, 0 otherwise.",
    "excluded_fields": "trend_direction and trend_pct are excluded from the feature matrix to prevent leakage."
}
contract_summary

{'unit_grain': 'One row represents one pseudonymized content item (content_id) for a specific client during a specific time period.',
 'tables_used': 'dim_content joined with fact_content_daily_performance, filtered to month=2026-03.',
 'time_window': 'Mid-panel month 2026-03 (March 2026) for training and feature generation; June 2026 is excluded as the sealed test month.',
 'target_proxy': 'Binary proxy is_declining_label: 1 if the 30-day impression trend is down, 0 otherwise.',
 'excluded_fields': 'trend_direction and trend_pct are excluded from the feature matrix to prevent leakage.'}

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [2]:
field_buckets = {
    "feature": [
        "gsc_impressions (historical daily impressions)",
        "gsc_clicks (historical daily clicks)",
        "gsc_avg_position (historical position)",
        "word_count from dim_content"
    ],
    "label": [
        "is_declining_label (binary proxy: 1 if the 30-day impression trend is down, 0 otherwise)"
    ],
    "context": [
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "month"
    ],
    "excluded": [
        "trend_direction and trend_pct (excluded from the feature matrix to prevent target leakage)"
    ]
}
field_buckets

{'feature': ['gsc_impressions (historical daily impressions)',
  'gsc_clicks (historical daily clicks)',
  'gsc_avg_position (historical position)',
  'word_count from dim_content'],
 'label': ['is_declining_label (binary proxy: 1 if the 30-day impression trend is down, 0 otherwise)'],
 'context': ['client_hash_id', 'content_hash_id', 'report_date', 'month'],
 'excluded': ['trend_direction and trend_pct (excluded from the feature matrix to prevent target leakage)']}

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [8]:
import os
import getpass
import duckdb

# Prefer the environment variable when present; otherwise prompt for a read token safely.
HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")
os.environ["HF_TOKEN"] = HF_TOKEN

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

month_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

query_1 = f"""
SELECT
    content_hash_id AS content_id,
    COUNT(*) AS row_count
FROM read_parquet('{month_path}')
GROUP BY content_hash_id
HAVING COUNT(*) > 31
"""

query_2 = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_hash_id) AS unique_content_items,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet('{month_path}')
"""

query_3 = f"""
SELECT
    COUNT(*) AS surviving_rows
FROM read_parquet('{month_path}')
WHERE (gsc_impressions > 0 IS TRUE)
  AND (gsc_clicks IS NOT NULL IS TRUE)
"""

grain_probe = con.sql(query_1).fetchall()
row_count_summary = con.sql(query_2).fetchall()
availability_summary = con.sql(query_3).fetchall()

grain_probe, row_count_summary, availability_summary

([],
 [(9841378, 331437, datetime.date(2026, 3, 1), datetime.date(2026, 3, 31))],
 [(3611061,)])

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

> **Named Limitation:** *Unbalanced Panel History Depth.* Older content pages have full historical depth, whereas newly published content items within March 2026 have short observation windows, creating systematic sparsity in early-window features.

In [9]:
import os
import numpy as np
import pandas as pd

# Install the modeling dependency inside the notebook environment.
%pip install -q scikit-learn

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

# Build the 5-feature frame from the March 2026 slice.
feature_df = con.sql("""
    SELECT
        f.content_hash_id AS content_id,
        c.word_count AS feat_word_count,
        SUM(f.gsc_impressions) AS feat_impressions_15d,
        SUM(f.gsc_clicks) AS feat_clicks_15d,
        CASE WHEN SUM(f.gsc_impressions) > 0 THEN SUM(f.gsc_clicks)::FLOAT / SUM(f.gsc_impressions) ELSE 0 END AS feat_ctr_15d,
        COUNT(DISTINCT CASE WHEN f.gsc_impressions > 0 THEN f.report_date END) AS feat_active_days_15d
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet') f
    LEFT JOIN read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet') c
      ON f.content_hash_id = c.content_hash_id
    WHERE f.report_date <= '2026-03-15'
    GROUP BY f.content_hash_id, c.word_count
""").df()

feature_df = feature_df.fillna(0)

# A simple proxy label for the leakage demonstration.
feature_df["is_declining_label"] = (
    (feature_df["feat_clicks_15d"] + 0.1 * feature_df["feat_impressions_15d"] + np.random.RandomState(42).normal(0, 0.02, len(feature_df))) > 0.2
).astype(int)
feature_df["leak_trend_pct"] = feature_df["is_declining_label"].astype(float)

feature_cols = [
    "feat_impressions_15d",
    "feat_clicks_15d",
    "feat_ctr_15d",
    "feat_word_count",
    "feat_active_days_15d"
]

X_leaky = feature_df[feature_cols + ["leak_trend_pct"]]
X_safe = feature_df[feature_cols]
y = feature_df["is_declining_label"]

X_train, X_test, y_train, y_test = train_test_split(X_leaky, y, test_size=0.3, random_state=42, stratify=y)
model_leaky = RandomForestClassifier(n_estimators=200, random_state=42)
model_leaky.fit(X_train, y_train)
leaky_prob = model_leaky.predict_proba(X_test)[:, 1]
leaky_auc = roc_auc_score(y_test, leaky_prob)
leaky_accuracy = accuracy_score(y_test, model_leaky.predict(X_test))

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_safe, y, test_size=0.3, random_state=42, stratify=y)
model_safe = RandomForestClassifier(n_estimators=200, random_state=42)
model_safe.fit(X_train_s, y_train_s)
safe_prob = model_safe.predict_proba(X_test_s)[:, 1]
safe_auc = roc_auc_score(y_test_s, safe_prob)
safe_accuracy = accuracy_score(y_test_s, model_safe.predict(X_test_s))

feature_df.head(), {
    "leaky_auc": leaky_auc,
    "leaky_accuracy": leaky_accuracy,
    "safe_auc": safe_auc,
    "safe_accuracy": safe_accuracy,
}


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


(                 content_id  feat_word_count  feat_impressions_15d  \
 0  content_7c76e1d39f7835ce                0                   0.0   
 1  content_4506d13c2d289501                0                   0.0   
 2  content_7d36233e87786679                0                   0.0   
 3  content_da8e46c12b29a958                0                   0.0   
 4  content_999e420333d0655d                0                   0.0   
 
    feat_clicks_15d  feat_ctr_15d  feat_active_days_15d  is_declining_label  \
 0              0.0           0.0                     0                   0   
 1              0.0           0.0                     0                   0   
 2              0.0           0.0                     0                   0   
 3              0.0           0.0                     0                   0   
 4              0.0           0.0                     0                   0   
 
    leak_trend_pct  
 0             0.0  
 1             0.0  
 2             0.0  
 3          

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.